In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [7]:
import pandas as pd
from scipy import sparse
import torch

In [8]:
from Utils import *
from Model import *

# LGG

In [5]:
cancer = "LGG"

In [9]:
data = pd.read_csv(f"/home/koe3/Bioinformatics/Data/{cancer}/0220_LGG_Normed_Entire_Data.csv")
x = torch.from_numpy(data.values).to(dtype = torch.float)

In [10]:
data

,HMGB1P1,A2M,AACS,AADAT,AANAT,AARS2,AASDHPPT,AASDH,AASS,ABAT,...,ZIC2,ZMAT2,ZMAT3,ZNF274,ZNRF3,ZW10,ZWILCH,ZWINT,ZYX,Sex
0,1.777716,0.389232,-1.177514,0.517646,-0.527263,0.475746,-1.120785,0.469608,-0.263035,0.845147,...,0.837401,0.514185,-0.578702,1.071558,0.765242,-0.439994,-1.164296,-1.448524,-0.809574,0
1,0.233934,0.707874,-0.794603,-0.908896,0.586772,-0.776142,-0.485973,-0.160776,0.959209,0.646833,...,1.963239,-0.155367,0.486913,-0.527983,0.277609,-0.035972,0.323295,-0.505197,0.818143,1
2,0.711468,0.824952,-0.271857,1.699544,-0.146400,-1.522602,0.322414,-0.092264,-1.469562,-0.260285,...,-1.301798,0.803925,0.508958,-0.842355,-0.304808,0.249729,-0.458007,0.128614,-0.213299,0
3,1.791092,0.007414,-0.303279,-0.655909,-1.619284,0.736664,0.497619,-1.197012,-1.604390,-0.459375,...,-1.647006,0.889620,-0.325959,1.012508,-1.208278,1.686594,2.562832,2.373111,-0.283270,1
4,1.157604,-0.112626,-0.342409,-1.595685,-0.101821,-0.458447,-0.456942,-0.001335,-2.456418,0.024421,...,0.952211,1.766846,-0.962883,-1.490118,0.266920,-0.494128,-1.748391,-1.310929,-0.705312,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,-0.399459,-1.481720,2.889208,-0.322603,-0.221191,1.158866,0.301227,-0.936558,0.256070,-0.789024,...,-0.383512,-0.226762,-1.715259,-1.081246,0.072150,-0.608469,0.286122,0.132115,-0.470074,0
242,-0.413783,-0.370962,-0.024195,1.461690,0.159579,-0.219504,-0.095633,0.518314,0.278427,0.283403,...,0.264129,0.068849,-0.081157,1.076537,0.171947,-1.680221,-0.745575,-0.659906,-0.929458,1
243,-0.218842,0.715807,-0.877743,1.792280,-0.769873,0.728809,0.759816,0.728172,0.676274,-0.173310,...,-0.776744,0.608524,-1.197759,1.242038,-1.331913,1.028516,0.769277,0.889907,-0.837304,1
244,-0.506558,-0.106510,-0.508298,1.856691,0.339280,0.724135,1.219016,0.623828,0.863336,0.192061,...,1.101532,0.308432,-1.414768,2.049526,0.537743,2.332232,-0.836792,-0.923038,-0.467301,1


In [16]:
sex_labels = data.iloc[:,-1]
sex_labels

0      0
1      1
2      0
3      1
4      1
      ..
241    0
242    1
243    1
244    1
245    1
Name: Sex, Length: 246, dtype: int64

In [17]:
x, x.shape

(tensor([[ 1.7777e+00,  3.8923e-01, -1.1775e+00,  ..., -1.4485e+00,
          -8.0957e-01,  0.0000e+00],
         [ 2.3393e-01,  7.0787e-01, -7.9460e-01,  ..., -5.0520e-01,
           8.1814e-01,  1.0000e+00],
         [ 7.1147e-01,  8.2495e-01, -2.7186e-01,  ...,  1.2861e-01,
          -2.1330e-01,  0.0000e+00],
         ...,
         [-2.1884e-01,  7.1581e-01, -8.7774e-01,  ...,  8.8991e-01,
          -8.3730e-01,  1.0000e+00],
         [-5.0656e-01, -1.0651e-01, -5.0830e-01,  ..., -9.2304e-01,
          -4.6730e-01,  1.0000e+00],
         [-8.4891e-01, -6.1385e-04,  2.3801e-01,  ..., -4.8876e-01,
          -6.9080e-01,  1.0000e+00]]),
 torch.Size([246, 4790]))

In [12]:
label = pd.read_csv(f"/home/koe3/Bioinformatics/Data/{cancer}/0220_LGG_Entire_Label.csv")
label

,Label
0,1
1,0
2,1
3,1
4,1
...,...
241,1
242,1
243,1
244,1


---

## CL Part

In [13]:
model_date="0813"
model_num=1

In [14]:
hparam_map = {
    1: ("Run_v19", "he_normal"),
    2: ("Run_v19", "he_normal"),
    3: ("Run_v19", "he_normal"),
    4: ("Run_v18", "he_uniform"),
    5: ("Run_v19", "he_normal"),
    6: ("Run_v18", "he_normal"),
    7: ("Run_v18", "he_uniform"),
    8: ("Run_v19", "he_normal"),
    9: ("Run_v35", "he_normal"),
    10: ("Run_v18", "he_uniform")
}

In [19]:
def get_num_nodes(data_name):
    """Returns the number of input features (genes) for a given dataset."""
    node_map = {
        "LIHC": 4781, "STAD": 4790, "LUAD": 4786,
        "LUSC": 4787, "LGG": 4789, "GSE240567": 4500
    }
    if data_name not in node_map:
        raise ValueError(f"Unknown dataset name for get_num_nodes: {data_name}")
        
    return node_map[data_name]

In [21]:
def load_sparse_indices(path):
    coo = sparse.load_npz(path)
    indices = np.vstack((coo.row, coo.col))
    return indices

In [22]:
sparse_indices = load_sparse_indices(f"/home/koe3/Bioinformatics/Data/TCGA_{cancer}_Pathway_Mask.npz")

In [26]:
avg_cmn_embeddings = torch.zeros((x.shape[0], 128))
avg_spf_embeddings = torch.zeros((x.shape[0], 128))
# print(avg_cmn_embeddings)
for experiment in range(1, 11):
    print("######################## %d experiment ########################" % (experiment))
    cl_net_hparams = [get_num_nodes(cancer), [395, 128, 16], 1, hparam_map[experiment][1], "sigmoid", 0.4] ### LGG
    cl_model = CLModule(cl_net_hparams, sparse_indices)
    
    # print({hparam_map[experiment][0]})
    cl_model.load_state_dict(torch.load(f"/home/koe3/nasdatafolder/Cluster/Bioinformatics/Proposed/SHCL_v3/{cancer}_Result/{hparam_map[experiment][0]}/Saved_Model/[{model_date}_{model_num}]_[{experiment}]Opt_Model_StateDict.pth", map_location='cpu'))
    cl_model.eval()
    with torch.no_grad():
        cmn_embeddings, _ = cl_model(x)       
        
        # Initialize a standard linear probe
        adversarial_probe = LogisticRegression(max_iter=1000)
        
        # Run 5-fold cross-validation to predict SEX from the COMMON embeddings
        # We use 'roc_auc' to see if the model can distinguish between males and females at all
        # print(cmn_embeddings)
        # print(sex_labels)
        scores = cross_val_score(adversarial_probe, cmn_embeddings[0], sex_labels, cv=5, scoring='roc_auc')

        mean_auroc = np.mean(scores)

        print(f"Adversarial Probe AUROC for predicting Sex: {mean_auroc:.3f}")


######################## 1 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.618
######################## 2 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.583
######################## 3 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.554
######################## 4 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.604
######################## 5 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.598
######################## 6 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.552
######################## 7 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.588
######################## 8 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.559
######################## 9 experiment ########################
Adversarial Probe AUROC for predi

# Asthma

In [27]:
asthma = "GSE240567"

In [43]:
data = pd.read_csv(f"/home/koe3/Bioinformatics/Data/Asthma/{asthma}/{asthma}_Normed_Entire_Data.csv")
x = torch.from_numpy(data.values).to(dtype = torch.float)

In [44]:
data

,DPM1,FGR,CFH,FUCA2,GCLC,NFYA,SEMA3F,CFTR,CYP51A1,RAD52,...,H4C2,H3C10,ADORA3,PIGY,H3C2,H3C3,NPBWR1,UGT1A3,UGT1A5,Sex
0,0.413739,1.117053,0.663595,0.959307,0.817929,-1.305526,0.014267,-1.029322,0.681319,0.005165,...,0.632746,0.385485,0.409453,0.418256,0.234392,0.318593,-0.503568,-1.492085,-0.873495,1
1,0.114446,2.170677,0.461724,0.591021,-2.020163,-0.397902,0.379442,-1.548556,0.913926,-0.313306,...,0.657341,0.958368,0.442234,-0.995636,1.222865,0.997287,0.258784,-0.428357,-1.576770,0
2,1.443780,1.129986,-0.535696,-0.468586,-2.786815,-0.762972,0.207087,1.027602,0.246339,1.840001,...,0.652082,0.694510,2.407472,-3.029751,0.959210,1.049761,-0.954080,1.776278,1.877062,1
3,-1.089564,0.071863,-1.647470,-0.268176,-0.097398,0.468977,0.671395,-0.511815,0.598581,-0.606560,...,0.404252,-0.110976,0.168137,-0.480853,0.024609,-0.031836,1.188082,0.572094,1.050699,0
4,0.358323,0.261149,0.664754,0.691456,-0.006163,-1.421467,0.422877,-0.179158,0.647699,-1.462079,...,-0.351118,0.102787,-0.446755,-0.083824,-0.357480,0.060902,-0.193603,0.347017,0.117870,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,-0.393063,0.563204,0.640517,1.364692,0.085180,-0.640820,0.066068,-1.513612,0.244345,-0.472774,...,0.333290,1.039323,0.406330,0.325758,1.597732,1.277206,-1.550051,-0.670255,-0.164198,0
532,0.084791,-0.307321,0.444163,-0.477689,0.647847,0.052377,0.197531,0.242334,0.533503,-1.280742,...,0.468889,0.463161,0.031190,1.450512,0.592012,0.678726,0.542118,-0.797493,-0.964889,0
533,1.185496,0.213295,0.928651,0.394510,1.295366,0.210903,-0.265704,-0.614320,-0.120929,-0.636251,...,0.138450,0.055977,-0.414633,1.724079,0.336815,0.412270,-0.193252,-0.467288,0.253874,1
534,0.230130,0.519622,0.341761,0.525099,0.611224,0.274294,0.092080,-0.153758,0.595299,-0.513446,...,0.559750,0.697311,0.089098,0.919559,0.859773,0.803443,-0.866965,-1.112102,-0.976648,0


In [45]:
sex_labels = data.iloc[:,-1]
sex_labels

0      1
1      0
2      1
3      0
4      1
      ..
531    0
532    0
533    1
534    0
535    0
Name: Sex, Length: 536, dtype: int64

In [46]:
x, x.shape

(tensor([[ 0.4137,  1.1171,  0.6636,  ..., -1.4921, -0.8735,  1.0000],
         [ 0.1144,  2.1707,  0.4617,  ..., -0.4284, -1.5768,  0.0000],
         [ 1.4438,  1.1300, -0.5357,  ...,  1.7763,  1.8771,  1.0000],
         ...,
         [ 1.1855,  0.2133,  0.9287,  ..., -0.4673,  0.2539,  1.0000],
         [ 0.2301,  0.5196,  0.3418,  ..., -1.1121, -0.9766,  0.0000],
         [-0.7269, -0.1076,  1.0011,  ..., -0.4520, -1.3123,  0.0000]]),
 torch.Size([536, 4501]))

---

## CL Part

In [32]:
model_ver="Retune_v65"
model_date="0806"
model_num=1

In [35]:
def get_num_nodes(data_name):
    """Returns the number of input features (genes) for a given dataset."""
    node_map = {
        "LIHC": 4781, "STAD": 4790, "LUAD": 4786,
        "LUSC": 4787, "LGG": 4789, "GSE240567": 4500
    }
    if data_name not in node_map:
        raise ValueError(f"Unknown dataset name for get_num_nodes: {data_name}")
        
    return node_map[data_name]

In [39]:
def load_sparse_indices(path):
    coo = sparse.load_npz(path)
    indices = np.vstack((coo.row, coo.col))
    return indices

In [41]:
sparse_indices = load_sparse_indices(f"/home/koe3/Bioinformatics/Data/Asthma/{asthma}/{asthma}_Gene_Pathway_Mask.npz")

In [47]:
# print(avg_cmn_embeddings)
for experiment in range(1, 11):
    print("######################## %d experiment ########################" % (experiment))
    cl_net_hparams = [get_num_nodes(asthma), [395, 128, 64], 1, "he_normal", "relu", 0.10070817185386628] ### GSE240567
    cl_model = CLModule(cl_net_hparams, sparse_indices)
    # print({hparam_map[experiment][0]})
    cl_model.load_state_dict(torch.load(f"/home/koe3/nasdatafolder/Cluster/Bioinformatics/Proposed/SHCL_v3/{asthma}_Result/{model_ver}/Saved_Model/[{model_date}_{model_num}]_[{experiment}]Opt_Model_StateDict.pth", map_location='cpu'))
    cl_model.eval()
    with torch.no_grad():
        cmn_embeddings, _ = cl_model(x)       
        
        # Initialize a standard linear probe
        adversarial_probe = LogisticRegression(max_iter=1000)
        
        # Run 5-fold cross-validation to predict SEX from the COMMON embeddings
        # We use 'roc_auc' to see if the model can distinguish between males and females at all
        # print(cmn_embeddings)
        # print(sex_labels)
        scores = cross_val_score(adversarial_probe, cmn_embeddings[0], sex_labels, cv=5, scoring='roc_auc')

        mean_auroc = np.mean(scores)

        print(f"Adversarial Probe AUROC for predicting Sex: {mean_auroc:.3f}")


######################## 1 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.611
######################## 2 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.656
######################## 3 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.564
######################## 4 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.639
######################## 5 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.606
######################## 6 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.604
######################## 7 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.609
######################## 8 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.620
######################## 9 experiment ########################
Adversarial Probe AUROC for predi

# LIHC

In [73]:
cancer = "LIHC"

In [74]:
data = pd.read_csv(f"/home/koe3/Bioinformatics/Data/{cancer}/LIHC_Normed_Entire_Data.csv")
x = torch.from_numpy(data.values).to(dtype = torch.float)

In [75]:
data

,HMGB1P1,A2M,AACS,AADAT,AANAT,AARS2,AASDHPPT,AASDH,AASS,ABAT,...,ZIC2,ZMAT2,ZMAT3,ZNF274,ZNRF3,ZW10,ZWILCH,ZWINT,ZYX,Sex
0,-0.565695,-0.214912,0.231044,0.328856,1.128280,0.293825,0.785618,0.077320,-0.174110,0.765913,...,0.356298,0.252973,-0.228853,-0.825474,0.382776,1.643824,1.857764,0.614014,0.702216,1
1,0.112144,-0.598789,-1.469801,1.249108,-0.612048,-0.646020,0.331313,0.149864,1.171912,0.865733,...,0.513432,-0.752117,0.708390,-0.035109,-0.466783,0.019952,-0.233835,-0.261080,-0.121487,1
2,-0.886980,1.472079,0.936094,-0.861659,-0.612048,-0.587517,0.155696,0.417658,0.623674,0.635868,...,0.057413,-1.154955,-0.459559,1.114756,-0.726070,0.270442,0.930542,0.243103,-0.187395,0
3,0.317415,-1.255813,-1.942794,1.455838,-0.612048,0.793340,0.358867,1.178859,0.136108,1.396449,...,-2.169207,-1.476569,0.504103,0.715247,-0.557678,1.479281,-0.721668,-1.426093,-0.292278,0
4,0.205466,0.064298,-0.010182,-0.622966,-0.612048,-0.669733,-0.329131,-0.733108,-0.330600,0.432251,...,0.484221,0.036803,0.989916,-0.784678,-0.373887,-0.365778,-1.075650,-0.548769,1.475375,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192,-0.762834,0.548766,1.324238,-0.156496,-0.612048,0.025409,0.609630,-0.707510,-0.174777,-0.722142,...,0.988406,-1.534299,0.854094,-0.971161,-0.119100,1.162346,1.259817,0.348537,0.253191,1
193,-0.782596,0.475609,-0.193588,-0.368832,-0.612048,-0.367886,0.056986,-1.107853,-0.347554,0.474605,...,-0.016851,-0.682549,-0.366288,-0.812695,-0.675572,-0.917522,-1.081681,-1.433368,0.248923,1
194,-0.767562,-0.600181,-1.083877,0.807619,-0.612048,1.140959,0.295782,0.197640,-0.011715,1.239959,...,0.008331,0.006937,-1.198362,0.071257,-0.118500,0.309544,-0.760675,-0.330917,0.254234,0
195,0.348334,-0.253379,-0.821124,1.732100,-0.612048,-1.267595,0.279639,0.917180,1.685385,1.723648,...,0.908557,-0.065727,1.804244,0.034196,-1.132650,0.839238,1.213806,1.313254,-1.788809,1


In [76]:
sex_labels = data.iloc[:,-1]
sex_labels

0      1
1      1
2      0
3      0
4      1
      ..
192    1
193    1
194    0
195    1
196    1
Name: Sex, Length: 197, dtype: int64

In [78]:
x, x.shape

(tensor([[-0.5657, -0.2149,  0.2310,  ...,  0.6140,  0.7022,  1.0000],
         [ 0.1121, -0.5988, -1.4698,  ..., -0.2611, -0.1215,  1.0000],
         [-0.8870,  1.4721,  0.9361,  ...,  0.2431, -0.1874,  0.0000],
         ...,
         [-0.7676, -0.6002, -1.0839,  ..., -0.3309,  0.2542,  0.0000],
         [ 0.3483, -0.2534, -0.8211,  ...,  1.3133, -1.7888,  1.0000],
         [ 0.4102, -0.0669, -1.8571,  ...,  1.3528, -2.8614,  1.0000]]),
 torch.Size([197, 4782]))

---

## CL Part

In [79]:
def get_num_nodes(data_name):
    """Returns the number of input features (genes) for a given dataset."""
    node_map = {
        "LIHC": 4781, "STAD": 4790, "LUAD": 4786,
        "LUSC": 4787, "LGG": 4789, "GSE240567": 4500
    }
    if data_name not in node_map:
        raise ValueError(f"Unknown dataset name for get_num_nodes: {data_name}")
        
    return node_map[data_name]

In [80]:
model_ver="Retune_v13"
model_date="0818"
model_num=1

In [82]:
cl_net_hparams = [get_num_nodes(cancer), [395, 128, 32], 1, "he_normal", "relu", 0.1] ### LIHC

In [83]:
sparse_indices = load_sparse_indices(f"/home/koe3/Bioinformatics/Data/TCGA_{cancer}_Pathway_Mask.npz")

In [84]:
# print(avg_cmn_embeddings)
for experiment in range(1, 11):
    print("######################## %d experiment ########################" % (experiment))
    cl_model = CLModule(cl_net_hparams, sparse_indices)
    # print({hparam_map[experiment][0]})
    cl_model.load_state_dict(torch.load(f"/home/koe3/nasdatafolder/Cluster/Bioinformatics/Proposed/SHCL_v3/{cancer}_Result/{model_ver}/Saved_Model/[{model_date}_{model_num}]_[{experiment}]Final_Model_StateDict.pth", map_location='cpu'))
    cl_model.eval()
    with torch.no_grad():
        cmn_embeddings, _ = cl_model(x)       
        
        # Initialize a standard linear probe
        adversarial_probe = LogisticRegression(max_iter=1000)
        
        # Run 5-fold cross-validation to predict SEX from the COMMON embeddings
        # We use 'roc_auc' to see if the model can distinguish between males and females at all
        # print(cmn_embeddings)
        # print(sex_labels)
        scores = cross_val_score(adversarial_probe, cmn_embeddings[0], sex_labels, cv=5, scoring='roc_auc')

        mean_auroc = np.mean(scores)

        print(f"Adversarial Probe AUROC for predicting Sex: {mean_auroc:.3f}")


######################## 1 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.711
######################## 2 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.521
######################## 3 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.599
######################## 4 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.710
######################## 5 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.521
######################## 6 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.580
######################## 7 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.707
######################## 8 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.553
######################## 9 experiment ########################
Adversarial Probe AUROC for predi

# LUAD

In [85]:
cancer = "LUAD"

In [90]:
data = pd.read_csv(f"/home/koe3/Bioinformatics/Data/{cancer}/LUAD_Normed_Entire_Data.csv")
x = torch.from_numpy(data.values).to(dtype = torch.float)

In [91]:
data

,HMGB1P1,A2M,AACS,AADAT,AANAT,AARS2,AASDHPPT,AASDH,AASS,ABAT,...,ZIC2,ZMAT2,ZMAT3,ZNF274,ZNRF3,ZW10,ZWILCH,ZWINT,ZYX,Sex
0,-0.317537,0.583572,0.397272,0.946074,-0.380023,-0.849356,0.281688,-0.041707,0.067527,0.655053,...,0.309801,-1.324163,0.890902,0.889411,0.445303,0.481898,-1.026628,-0.889713,-0.827177,1
1,-1.868705,0.064878,-0.212698,0.059370,0.135101,-0.170504,0.052477,0.035131,-0.643554,-1.127459,...,0.955615,0.378770,-0.125760,-0.708203,-0.474243,0.518065,0.943597,0.809503,0.830757,0
2,-1.117771,-0.899911,1.133315,0.164526,-0.357014,1.467315,-1.100320,0.533571,-1.454212,-0.095583,...,-0.670948,-0.597567,1.199742,0.465838,3.658665,-0.512876,0.470183,0.594787,-0.654757,1
3,0.167485,-0.159358,1.367650,1.261743,-1.006285,-0.318681,-0.247322,-1.002704,-0.534619,0.791189,...,-1.203090,-0.418747,-0.924923,-0.511816,-0.247665,-0.920913,-0.012347,1.233450,0.763813,0
4,-1.506211,-1.553305,-0.292601,-3.347270,1.366161,-1.631700,0.111465,0.410898,-2.415049,-0.333679,...,1.079244,0.390817,-0.865696,-1.370657,-1.639191,-0.428301,0.188187,0.237547,0.644715,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267,-0.671573,-1.010643,1.785873,-1.905787,2.439335,0.016307,0.031261,0.792503,-2.024470,-1.862423,...,-1.203090,1.152402,-1.192045,0.135916,1.062680,0.894108,-0.208563,-0.534922,-0.490706,0
268,1.215660,-0.387259,-0.653309,0.629200,0.715916,0.384077,-0.931287,-0.586747,0.834745,1.082248,...,1.400697,2.347892,-0.944632,0.674023,0.235197,-0.358657,-1.039005,-1.549926,0.093789,1
269,0.634537,0.238601,0.802856,1.085646,0.903640,-0.163644,-0.613544,-0.319777,-0.553141,-0.314514,...,1.962534,0.108392,-0.597374,1.760676,0.164508,-0.224190,-1.864937,-1.611344,0.550526,0
270,-0.559327,-0.154802,-0.133591,-0.467515,1.369332,0.305197,-0.320487,0.649545,-0.524881,0.753237,...,0.559174,1.034969,-0.645596,1.339191,-2.194311,0.457880,0.091470,0.003827,0.313985,0


In [92]:
sex_labels = data.iloc[:,-1]
sex_labels

0      1
1      0
2      1
3      0
4      1
      ..
267    0
268    1
269    0
270    0
271    0
Name: Sex, Length: 272, dtype: int64

In [93]:
x, x.shape

(tensor([[-0.3175,  0.5836,  0.3973,  ..., -0.8897, -0.8272,  1.0000],
         [-1.8687,  0.0649, -0.2127,  ...,  0.8095,  0.8308,  0.0000],
         [-1.1178, -0.8999,  1.1333,  ...,  0.5948, -0.6548,  1.0000],
         ...,
         [ 0.6345,  0.2386,  0.8029,  ..., -1.6113,  0.5505,  0.0000],
         [-0.5593, -0.1548, -0.1336,  ...,  0.0038,  0.3140,  0.0000],
         [-1.3288,  0.6065,  0.5490,  ..., -0.5640,  0.4491,  0.0000]]),
 torch.Size([272, 4787]))

---

## CL Part

In [94]:
model_date="0917"
model_num=1

In [95]:
hparam_map = {
    1: ("Run_v19", "he_normal", 0.08356555216558127),
    2: ("Run_v19", "he_normal", 0.08356555216558127),
    3: ("Run_v19", "he_normal", 0.08356555216558127),
    4: ("Run_v18", "he_uniform", 0.12032037668773907),
    5: ("Run_v19", "he_normal", 0.07581217434771849),
    6: ("Run_v18", "he_normal", 0.12032037668773907),
    7: ("Run_v18", "he_uniform", 0.12032037668773907),
    8: ("Run_v19", "he_normal", 0.08356555216558127),
    9: ("Run_v35", "he_normal", 0.07581217434771849), ###clf-6
    10: ("Run_v18", "he_uniform", 0.12032037668773907)
    }

In [97]:
cl_net_hparams = [get_num_nodes(cancer), [395, 128, 64], 1, "he_normal", "relu", 0.1] ### LUAD

In [98]:
sparse_indices = load_sparse_indices(f"/home/koe3/Bioinformatics/Data/TCGA_{cancer}_Pathway_Mask.npz")

In [99]:
# print(avg_cmn_embeddings)
for experiment in range(1, 11):
    print("######################## %d experiment ########################" % (experiment))
    cl_model = CLModule(cl_net_hparams, sparse_indices)
    # print({hparam_map[experiment][0]})    
    cl_model.load_state_dict(torch.load(f"/home/koe3/nasdatafolder/Cluster/Bioinformatics/Proposed/SHCL_v3/{cancer}_Result/Retune_v33/Saved_Model/[{model_date}_{model_num}]_[{experiment}]Final_Model_StateDict.pth", map_location='cpu'))
    cl_model.eval()
    with torch.no_grad():
        cmn_embeddings, _ = cl_model(x)       
        
        # Initialize a standard linear probe
        adversarial_probe = LogisticRegression(max_iter=1000)
        
        # Run 5-fold cross-validation to predict SEX from the COMMON embeddings
        # We use 'roc_auc' to see if the model can distinguish between males and females at all
        # print(cmn_embeddings)
        # print(sex_labels)
        scores = cross_val_score(adversarial_probe, cmn_embeddings[0], sex_labels, cv=5, scoring='roc_auc')

        mean_auroc = np.mean(scores)

        print(f"Adversarial Probe AUROC for predicting Sex: {mean_auroc:.3f}")


######################## 1 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.552
######################## 2 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.498
######################## 3 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.514
######################## 4 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.554
######################## 5 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.543
######################## 6 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.516
######################## 7 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.560
######################## 8 experiment ########################
Adversarial Probe AUROC for predicting Sex: 0.501
######################## 9 experiment ########################
Adversarial Probe AUROC for predi